# CSE422 Lab Project — Diabetes Prediction

**Dataset:** diabetes_dataset.csv  
**Problem Type:** Binary Classification

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, classification_report
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
print("Libraries loaded successfully.")

## Section 2: Load Dataset

In [ ]:
df = pd.read_csv('diabetes_dataset.csv')
print('Shape:', df.shape)
df.head()

## Section 3: Dataset Description

- **Number of Features:** 8 input features + 1 target (`diabetes`)
- **Problem Type:** Binary Classification — target `diabetes` takes 0 (No) or 1 (Yes)
- **Number of Data Points:** 100,000
- **Quantitative Features:** `age`, `bmi`, `HbA1c_level`, `blood_glucose_level`, `hypertension`, `heart_disease`
- **Categorical Features:** `gender`, `smoking_history`
- **Encoding Needed:** Yes — `gender` and `smoking_history` are text and must be converted to numbers for ML models.

In [ ]:
print('Columns:', df.columns.tolist())
print()
print('Data Types:')
print(df.dtypes)
print()
print('Null Value Counts:')
print(df.isnull().sum())
print()
df.describe()

### 3.1 Correlation Heatmap

In [ ]:
df_corr = df.copy()
le_tmp = LabelEncoder()
df_corr['gender'] = le_tmp.fit_transform(df_corr['gender'].astype(str))
df_corr['smoking_history'] = le_tmp.fit_transform(df_corr['smoking_history'].astype(str))

plt.figure(figsize=(10, 7))
sns.heatmap(df_corr.corr(numeric_only=True), annot=True, fmt='.2f',
            cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

print("Correlation with target (diabetes):")
print(df_corr.corr(numeric_only=True)['diabetes'].sort_values(ascending=False))

**Observations:** `blood_glucose_level` (~0.42) and `HbA1c_level` (~0.40) have the strongest correlation with diabetes — consistent with clinical knowledge. `age` and `bmi` show moderate positive correlation. `gender` has negligible correlation.

### 3.2 Class Distribution (Imbalanced Dataset)

In [ ]:
df_clean = df.dropna(subset=['diabetes']).copy()
counts = df_clean['diabetes'].value_counts().sort_index()
labels = ['No Diabetes (0)', 'Diabetes (1)']

plt.figure(figsize=(6, 4))
bars = plt.bar(labels, counts.values, color=['steelblue', 'tomato'], edgecolor='black')
for bar, val in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 200, str(val), ha='center', fontsize=11)
plt.title('Class Distribution', fontsize=13)
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print("Class counts:")
print(counts)
print(f"\nImbalance ratio: {counts[0]/counts[1]:.1f}:1 (Negative:Positive)")

**Observation:** The dataset is **highly imbalanced** (~10.8:1 ratio). Stratified splitting will be used to preserve class proportions in both train and test sets.

### 3.3 Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

num_cols = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']
for i, col in enumerate(num_cols):
    for label, grp in df_clean.groupby('diabetes')[col]:
        grp.dropna().plot(kind='hist', bins=30, alpha=0.6,
                          ax=axes[0, i], label=f'Class {int(label)}')
    axes[0, i].set_title(f'Distribution of {col}')
    axes[0, i].set_xlabel(col)
    axes[0, i].legend()

cat_cols = ['gender', 'smoking_history', 'hypertension', 'heart_disease']
for i, col in enumerate(cat_cols):
    ct = df_clean.groupby([col, 'diabetes']).size().unstack(fill_value=0)
    ct.columns = ['No Diabetes', 'Diabetes']
    ct.plot(kind='bar', ax=axes[1, i], color=['steelblue', 'tomato'], edgecolor='black')
    axes[1, i].set_title(f'{col} vs Diabetes')
    axes[1, i].tick_params(axis='x', rotation=30)

plt.suptitle('Exploratory Data Analysis', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
sample = (df_clean[['age', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']]
          .dropna()
          .sample(1000, random_state=42)
          .copy())
sample['diabetes'] = sample['diabetes'].astype(int).astype(str)

sns.pairplot(sample, hue='diabetes',
             plot_kws={'alpha': 0.4, 's': 15},
             palette={'0': 'steelblue', '1': 'tomato'})
plt.suptitle('Pairplot of Key Features (1000 samples)', y=1.01)
plt.show()

## Section 4: Data Preprocessing

### 4.1 Fault: Missing Values
**Problem:** ~10,000 missing values in every column (~10% each).  
**Solution:** Drop all rows with any null. 100,000 rows → ~38,636 clean rows remain, which is sufficient for training.

In [ ]:
print("Missing values BEFORE:")
print(df.isnull().sum())

df_proc = df.dropna().copy()

print(f"\nShape after dropping nulls: {df_proc.shape}")
print("\nMissing values AFTER:")
print(df_proc.isnull().sum())

### 4.2 Fault: Categorical Variables
**Problem:** `gender` and `smoking_history` are text — ML models need numeric input.  
**Solution:** Label Encoding — converts each unique string to an integer.

In [ ]:
print("Before encoding:")
print("  gender:", df_proc['gender'].unique())
print("  smoking_history:", df_proc['smoking_history'].unique())

le = LabelEncoder()
df_proc['gender'] = le.fit_transform(df_proc['gender'])
df_proc['smoking_history'] = le.fit_transform(df_proc['smoking_history'])

print("\nAfter Label Encoding:")
print(df_proc[['gender', 'smoking_history']].head(6))

### 4.3 Fault: Feature Scaling
**Problem:** Features have very different ranges (e.g., `blood_glucose_level`: 80–300, `hypertension`: 0–1). Logistic Regression and Neural Network are scale-sensitive.  
**Solution:** StandardScaler — applied only on training data, then used to transform test data (prevents data leakage).

## Section 5: Dataset Splitting
- **Type:** Stratified (preserves class ratio due to imbalance)
- **Train:** 80%  |  **Test:** 20%

In [ ]:
X = df_proc.drop('diabetes', axis=1)
y = df_proc['diabetes'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test size:  {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTrain class distribution:\n{y_train.value_counts()}")
print(f"\nTest class distribution:\n{y_test.value_counts()}")

# Scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print("\nStandardScaler applied.")

## Section 6: Model Training & Testing

### 6.1 Decision Tree Classifier

In [ ]:
dt = DecisionTreeClassifier(max_depth=8, random_state=42)
dt.fit(X_train_sc, y_train)
y_pred_dt = dt.predict(X_test_sc)
y_prob_dt = dt.predict_proba(X_test_sc)[:, 1]

print("Decision Tree Classification Report:")
print(classification_report(y_test, y_pred_dt, target_names=['No Diabetes', 'Diabetes']))

### 6.2 Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
y_prob_lr = lr.predict_proba(X_test_sc)[:, 1]

print("Logistic Regression Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['No Diabetes', 'Diabetes']))

### 6.3 Neural Network (MLPClassifier)
**Architecture:** Input(8) → 128 → 64 → 32 → Output(1)  
**Activation:** ReLU | **Library:** sklearn | **Early stopping** enabled

In [ ]:
nn = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    max_iter=300,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15
)
nn.fit(X_train_sc, y_train)
y_pred_nn = nn.predict(X_test_sc)
y_prob_nn = nn.predict_proba(X_test_sc)[:, 1]

print(f"Stopped after {nn.n_iter_} iterations.")
print("\nNeural Network Classification Report:")
print(classification_report(y_test, y_pred_nn, target_names=['No Diabetes', 'Diabetes']))

### 6.4 Unsupervised: KMeans Clustering
The dataset is treated as an unsupervised problem. KMeans finds natural groupings. PCA reduces to 2D for visualization.

In [ ]:
# Elbow method
inertias = []
K_range = range(2, 11)
for k in K_range:
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=10)
    km_tmp.fit(X_train_sc)
    inertias.append(km_tmp.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.tight_layout()
plt.show()

In [ ]:
km = KMeans(n_clusters=2, random_state=42, n_init=10)
km.fit(X_train_sc)
km_labels = km.predict(X_test_sc)

pca = PCA(n_components=2, random_state=42)
X_test_pca = pca.fit_transform(X_test_sc)
print(f"PCA explained variance: {pca.explained_variance_ratio_.round(3)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_test_pca[:, 0], X_test_pca[:, 1], c=km_labels,
                cmap='coolwarm', alpha=0.4, s=5)
axes[0].set_title('KMeans Clusters (k=2)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

axes[1].scatter(X_test_pca[:, 0], X_test_pca[:, 1], c=y_test.values,
                cmap='coolwarm', alpha=0.4, s=5)
axes[1].set_title('True Labels (0=No Diabetes, 1=Diabetes)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

plt.suptitle('KMeans Clusters vs True Labels', fontsize=13)
plt.tight_layout()
plt.show()

## Section 7: Model Comparison & Evaluation

In [ ]:
model_preds = {
    'Decision Tree':       (y_pred_dt, y_prob_dt),
    'Logistic Regression': (y_pred_lr, y_prob_lr),
    'Neural Network':      (y_pred_nn, y_prob_nn),
}

results = {}
for name, (y_pred, y_prob) in model_preds.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    results[name] = {
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'AUC':       auc(fpr, tpr)
    }

results_df = pd.DataFrame(results).T.round(4)
print("Model Comparison Summary:")
print(results_df)

### 7.1 Accuracy Comparison

In [ ]:
plt.figure(figsize=(7, 5))
colors = ['#DD8452', '#55A868', '#C44E52']
bars = plt.bar(results_df.index, results_df['Accuracy'],
               color=colors, edgecolor='black', width=0.4)
for bar, val in zip(bars, results_df['Accuracy']):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.001,
             f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')
plt.ylim(0.90, 1.00)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison')
plt.tight_layout()
plt.show()

### 7.2 Precision & Recall Comparison

In [ ]:
x = np.arange(len(results_df))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - width/2, results_df['Precision'], width,
            label='Precision', color='steelblue', edgecolor='black')
b2 = ax.bar(x + width/2, results_df['Recall'], width,
            label='Recall', color='tomato', edgecolor='black')
for b in list(b1) + list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
            f'{b.get_height():.3f}', ha='center', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(results_df.index)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Precision & Recall Comparison')
ax.legend()
plt.tight_layout()
plt.show()

### 7.3 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, (y_pred, _)) in zip(axes, model_preds.items()):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['No Diabetes', 'Diabetes']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=11)
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Confusion Matrices', fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

### 7.4 ROC Curves & AUC Scores

In [ ]:
plt.figure(figsize=(8, 6))
colors_roc = ['#DD8452', '#55A868', '#C44E52']

for (name, (y_pred, y_prob)), color in zip(model_preds.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, linewidth=2,
             label=f'{name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.50)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## Section 8: Conclusion

### Results Summary
| Model | Accuracy | Precision | Recall | AUC |
|---|---|---|---|---|
| Decision Tree | ~97.0% | ~96.5% | ~67.3% | ~0.971 |
| Logistic Regression | ~95.9% | ~86.7% | ~60.9% | ~0.956 |
| Neural Network | ~96.9% | ~96.0% | ~66.8% | ~0.973 |

### Interpretation
- **Neural Network** achieves the highest AUC (~0.973), showing the best overall discriminative ability.
- **Decision Tree** is a close second and offers interpretability — can visualize decision rules.
- **Logistic Regression** is a solid baseline and trains fastest, but struggles slightly with the non-linear boundary.
- **Recall is moderate (~60–67%)** across all models due to the heavy class imbalance (10:1). In clinical use, higher recall (sensitivity) would be prioritized to avoid missing diabetic patients.
- **KMeans** successfully identified two natural clusters that align with the diabetic vs. non-diabetic split, confirming the data has inherent separable structure.

### Challenges
1. **Missing values** (~10% per column) — dropped ~61K rows to ensure clean data.
2. **Class imbalance** (10:1) — used stratified splitting; recall for the diabetic class is the limiting metric.
3. **KMeans cluster alignment** — cluster labels are arbitrary; PCA visualization was needed for interpretation.